In [2]:
pip install FlagEmbedding

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 7.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 13.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 17.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.5/553.5 kB 12.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.0/36.0 MB 17.2 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 16.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 18.9 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 8.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.2 MB/s  0:00:00
  Attempting uninstall: hf-xet━━━━━━━━━━━━━━━━━━━━━━━━━━━  7/23 [lxml]ow]
    Found existing installation: hf-xet 1.5.2━━━━━━━━━━━━━━━━━  7/23 [lxml]
    Uninstalling hf-xet-1.5.2:━━━━━━━━━━━━━━━━━━━━━━━━━━━  7/23 [lxml]
      Successfully uninstalled hf-xet-1.5.2━━━━━━━━━━━━━━━━━━━  7/2

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = '/Users/natasyays/Downloads/Documents/SEMPRO/bali dataset/Preprocessing/dataset_final_sentences_english.csv'

df = pd.read_csv(DATA_PATH, usecols=["sentence"])
text_data = df["sentence"].fillna("").tolist()
print(f"Kalimat: {len(text_data):,}")

import torch
from FlagEmbedding import BGEM3FlagModel
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=torch.cuda.is_available())
output = model.encode(
    text_data,
    max_length=8192,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False
)
dense_embeddings = output['dense_vecs']
np.save("bali_embed_bge_m3.npy", dense_embeddings)
print(f"Tersimpan: {dense_embeddings.shape}")

In [1]:
import pandas as pd
import numpy as np
import torch
import time  # Untuk menghitung durasi
from tqdm import tqdm  # Untuk menampilkan progress bar & estimasi waktu
from FlagEmbedding import BGEM3FlagModel

DATA_PATH = '/Users/natasyays/Downloads/Documents/SEMPRO/bali dataset/Preprocessing/dataset_final_sentences_english.csv'

df = pd.read_csv(DATA_PATH, usecols=["sentence"])
text_data = df["sentence"].fillna("").tolist()
print(f"Total Kalimat: {len(text_data):,}")

# Deteksi GPU Mac (MPS)
if torch.backends.mps.is_available():
    device = "mps"
    print("Menggunakan GPU Mac (MPS).")
else:
    device = "cpu"
    print("Menggunakan CPU.")

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=False, device=device)

# Pengaturan batch
batch_size = 4  # Proses 4 kalimat sekaligus agar RAM/VRAM M1 tidak penuh
all_embeddings = []

print("\nMulai proses embedding...")
start_time = time.time()  # Catat waktu mulai

# Looping dengan tqdm untuk melihat Progress Bar dan ETA (Estimasi Waktu)
for i in tqdm(range(0, len(text_data), batch_size), desc="Memproses Data"):
    batch_text = text_data[i : i + batch_size]
    
    output = model.encode(
        batch_text,
        max_length=8192,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False
    )
    all_embeddings.append(output['dense_vecs'])

# Menggabungkan hasil batch menjadi satu array utuh
dense_embeddings = np.concatenate(all_embeddings, axis=0)

end_time = time.time()  # Catat waktu selesai
total_detik = end_time - start_time
total_menit = total_detik / 60
total_jam = total_menit / 60

print(f"\nSelesai! Total waktu eksekusi: {total_jam:.2f} Jam ({total_menit:.2f} Menit).")

np.save("bali_embed_bge_m3.npy", dense_embeddings)
print(f"Data tersimpan dengan dimensi: {dense_embeddings.shape}")

Total Kalimat: 72,374
Menggunakan GPU Mac (MPS).


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Mulai proses embedding...


Memproses Data: 100%|██████████| 18094/18094 [1:36:56<00:00,  3.11it/s]   



Selesai! Total waktu eksekusi: 1.62 Jam (97.08 Menit).
Data tersimpan dengan dimensi: (72374, 1024)
